# ML-07: Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order**: each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Pages with at least 500 March impressions and momentum at or below -20% enter the review queue. The score is the size of the momentum loss.

- **Reason code:** visible_recent_momentum_loss
- **Action:** review_for_refresh
- **Feature window:** March 2 to 31, 2026
- **Outcome window:** April 1 to 30, 2026


In [1]:
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
from huggingface_hub import get_token

token = get_token()
if not token:
    raise RuntimeError("Hugging Face READ token required.")
con = duckdb.connect()
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)", [token])

REL = "hf://datasets/FlyRank/internship-warehouse"
MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
APRIL = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"

feature_frame = con.sql(f"""
WITH f AS (
    SELECT client_hash_id, content_hash_id,
        SUM(gsc_impressions) AS impressions_feature_30d,
        SUM(gsc_impressions) FILTER (WHERE report_date <= DATE '2026-03-16') AS early,
        SUM(gsc_impressions) FILTER (WHERE report_date >= DATE '2026-03-17') AS late
    FROM {MARCH}
    WHERE gsc_data_available IS TRUE
      AND report_date BETWEEN DATE '2026-03-02' AND DATE '2026-03-31'
    GROUP BY 1, 2
    HAVING COUNT(DISTINCT report_date) = 30 AND SUM(gsc_impressions) >= 100
)
SELECT f.client_hash_id, f.content_hash_id, f.impressions_feature_30d,
    100.0 * (f.late - f.early) / f.early AS momentum_pct
FROM f
WHERE f.early > 0
""").df()

target_frame = con.sql(f"""
SELECT client_hash_id, content_hash_id,
    SUM(gsc_impressions) AS target_impressions
FROM {APRIL}
WHERE gsc_data_available IS TRUE
  AND report_date BETWEEN DATE '2026-04-01' AND DATE '2026-04-30'
GROUP BY 1, 2
HAVING COUNT(DISTINCT report_date) = 30
""").df()

evaluation_frame = feature_frame.merge(
    target_frame, on=["client_hash_id", "content_hash_id"]
)
evaluation_frame["is_declining_label"] = (
    evaluation_frame["target_impressions"]
    < 0.80 * evaluation_frame["impressions_feature_30d"]
).astype(int)

def bucket_table(values, bins, labels):
    table = (evaluation_frame.assign(bucket=pd.cut(values, bins, labels=labels))
             .groupby("bucket", observed=False)["is_declining_label"]
             .agg(n="size", declining_n="sum", rate="mean").reset_index())
    table["decline_rate_pct"] = (100 * table.pop("rate")).round(1)
    return table

print(f"March feature rows: {len(feature_frame)}")
print(f"Evaluation rows: {len(evaluation_frame)}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March feature rows: 62397
Evaluation rows: 50641


### Signal check 1: feature-window volume

**Question.** Does March impression volume track April decline?

**Verdict: MIXED**


In [2]:
volume_table = bucket_table(
    evaluation_frame["impressions_feature_30d"],
    [99, 499, 2_999, 29_999, np.inf],
    ["100-499", "500-2,999", "3,000-29,999", "30,000+"],
)
display(volume_table)

,bucket,n,declining_n,decline_rate_pct
0,100-499,7413,2464,33.2
1,"500-2,999",24934,13054,52.4
2,"3,000-29,999",17455,8121,46.5
3,"30,000+",839,337,40.2


Decline rates by ascending volume bucket are 33.2%, 52.4%, 46.5%, and 40.2%. The rule uses 500 impressions as its queue threshold.


### Signal check 2: prior-window momentum

**Question.** Does the change between the two March 15-day windows track April decline?

**Verdict: CONFIRMED**


In [3]:
momentum_table = bucket_table(
    evaluation_frame["momentum_pct"],
    [-np.inf, -50, -20, 0, 20, np.inf],
    ["<= -50%", "(-50%, -20%]", "(-20%, 0%]", "(0%, 20%]", "> 20%"],
)
display(momentum_table)

,bucket,n,declining_n,decline_rate_pct
0,<= -50%,4171,3375,80.9
1,"(-50%, -20%]",11439,6948,60.7
2,"(-20%, 0%]",9976,4921,49.3
3,"(0%, 20%]",8398,3428,40.8
4,> 20%,16657,5304,31.8


The decline rate falls from 80.9% for losses of at least 50% to 31.8% for gains above 20%.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

The queue contains pages that match the rule.


In [4]:
match = feature_frame["impressions_feature_30d"].ge(500) & feature_frame["momentum_pct"].le(-20)
scored = feature_frame[["client_hash_id", "content_hash_id",
                "impressions_feature_30d", "momentum_pct"]].copy()
scored["baseline_action_score"] = (-feature_frame["momentum_pct"]).where(match, 0)

queue = (scored[match]
         .sort_values(["baseline_action_score", "impressions_feature_30d"],
                      ascending=[False, False])
         .reset_index(drop=True))
queue.insert(0, "baseline_rank", range(1, len(queue) + 1))
queue["reason_code"] = "visible_recent_momentum_loss"
queue["action_label"] = "review_for_refresh"

OUTPUT_PATH = Path("../outputs/baseline_action_score.csv")
OUTPUT_PATH.parent.mkdir(exist_ok=True)
queue.to_csv(OUTPUT_PATH, index=False)

labels = evaluation_frame[["client_hash_id", "content_hash_id", "is_declining_label"]]
evaluated = queue.merge(labels, on=["client_hash_id", "content_hash_id"])
for k in (10, 20, 50):
    print(f"Precision@{k}: {evaluated.head(k)['is_declining_label'].mean():.2f}")
print(f"Base rate: {evaluation_frame['is_declining_label'].mean():.1%}")
print(f"Candidates: {len(queue):,}")
print("Wrote: work/outputs/baseline_action_score.csv")
display(queue[["baseline_rank", "baseline_action_score", "reason_code",
               "action_label", "impressions_feature_30d", "momentum_pct"]].head(10))

Precision@10: 0.90
Precision@20: 0.90
Precision@50: 0.86


Base rate: 47.3%
Candidates: 15,103
Wrote: work/outputs/baseline_action_score.csv


,baseline_rank,baseline_action_score,reason_code,action_label,impressions_feature_30d,momentum_pct
0,1,99.894217,visible_recent_momentum_loss,review_for_refresh,54887.0,-99.894217
1,2,97.077321,visible_recent_momentum_loss,review_for_refresh,2782.0,-97.077321
2,3,96.451613,visible_recent_momentum_loss,review_for_refresh,2889.0,-96.451613
3,4,96.255010,visible_recent_momentum_loss,review_for_refresh,8283.0,-96.255010
4,5,96.125666,visible_recent_momentum_loss,review_for_refresh,9357.0,-96.125666
5,6,95.442820,visible_recent_momentum_loss,review_for_refresh,1216.0,-95.442820
6,7,95.218923,visible_recent_momentum_loss,review_for_refresh,2082.0,-95.218923
7,8,95.050722,visible_recent_momentum_loss,review_for_refresh,6828.0,-95.050722
8,9,94.665272,visible_recent_momentum_loss,review_for_refresh,8056.0,-94.665272
9,10,93.880273,visible_recent_momentum_loss,review_for_refresh,25456.0,-93.880273


Precision is measured on rows with an observable April outcome. The exported queue is built only from March signals.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

The action and reason code repeat because this baseline intentionally contains one rule. Confidence uses only March evidence. April is used only to identify retrospective false positives; otherwise the final note records a plausible reason the refresh action could still be wrong.

In [5]:
top20 = queue.head(20).merge(
    labels, on=["client_hash_id", "content_hash_id"], how="left"
)

top20["why_it_is_here"] = [
    f"Momentum {m:.1f}%; {n:,.0f} impressions."
    for m, n in zip(top20["momentum_pct"], top20["impressions_feature_30d"])
]

def confidence_note(row):
    n = row["impressions_feature_30d"]
    loss = -row["momentum_pct"]
    if n >= 30_000:
        return f"HIGH: {loss:.1f}% loss across {n:,.0f} impressions makes random noise less likely."
    if n >= 3_000:
        return f"MEDIUM: {loss:.1f}% loss across {n:,.0f} impressions, but only one short window."
    return f"LOW: {loss:.1f}% loss across {n:,.0f} impressions may be sensitive to volatility."

def wrong_note(row):
    n = row["impressions_feature_30d"]
    if row["is_declining_label"] == 0:
        return "Observed weak pick: April did not meet the decline proxy, so the March loss did not persist."
    if n >= 30_000:
        return f"Even at {n:,.0f} impressions, a site-wide demand or tracking shock could make a page refresh ineffective."
    if n >= 3_000:
        return f"At {n:,.0f} impressions, seasonality, a SERP change, or demand shifting to another page could explain the loss."
    return f"At {n:,.0f} impressions, an early-window spike or ordinary volatility could exaggerate the loss."

top20["confidence"] = top20.apply(confidence_note, axis=1)
top20["what_would_make_it_wrong"] = top20.apply(wrong_note, axis=1)
display(top20[["baseline_rank", "action_label", "reason_code", "why_it_is_here",
               "confidence", "what_would_make_it_wrong"]])

,baseline_rank,action_label,reason_code,why_it_is_here,confidence,what_would_make_it_wrong
0,1,review_for_refresh,visible_recent_momentum_loss,"Momentum -99.9%; 54,887 impressions.","HIGH: 99.9% loss across 54,887 impressions mak...","Even at 54,887 impressions, a site-wide demand..."
1,2,review_for_refresh,visible_recent_momentum_loss,"Momentum -97.1%; 2,782 impressions.","LOW: 97.1% loss across 2,782 impressions may b...","At 2,782 impressions, an early-window spike or..."
2,3,review_for_refresh,visible_recent_momentum_loss,"Momentum -96.5%; 2,889 impressions.","LOW: 96.5% loss across 2,889 impressions may b...","At 2,889 impressions, an early-window spike or..."
3,4,review_for_refresh,visible_recent_momentum_loss,"Momentum -96.3%; 8,283 impressions.","MEDIUM: 96.3% loss across 8,283 impressions, b...","At 8,283 impressions, seasonality, a SERP chan..."
4,5,review_for_refresh,visible_recent_momentum_loss,"Momentum -96.1%; 9,357 impressions.","MEDIUM: 96.1% loss across 9,357 impressions, b...","At 9,357 impressions, seasonality, a SERP chan..."
5,6,review_for_refresh,visible_recent_momentum_loss,"Momentum -95.4%; 1,216 impressions.","LOW: 95.4% loss across 1,216 impressions may b...","At 1,216 impressions, an early-window spike or..."
6,7,review_for_refresh,visible_recent_momentum_loss,"Momentum -95.2%; 2,082 impressions.","LOW: 95.2% loss across 2,082 impressions may b...","At 2,082 impressions, an early-window spike or..."
7,8,review_for_refresh,visible_recent_momentum_loss,"Momentum -95.1%; 6,828 impressions.","MEDIUM: 95.1% loss across 6,828 impressions, b...","At 6,828 impressions, seasonality, a SERP chan..."
8,9,review_for_refresh,visible_recent_momentum_loss,"Momentum -94.7%; 8,056 impressions.","MEDIUM: 94.7% loss across 8,056 impressions, b...","At 8,056 impressions, seasonality, a SERP chan..."
9,10,review_for_refresh,visible_recent_momentum_loss,"Momentum -93.9%; 25,456 impressions.","MEDIUM: 93.9% loss across 25,456 impressions, ...","At 25,456 impressions, seasonality, a SERP cha..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Rows that miss the April proxy are shown below. The score uses only March impressions and March momentum; April is used only for evaluation.


In [6]:
display(top20.loc[top20["is_declining_label"].eq(0),
                  ["baseline_rank", "impressions_feature_30d", "momentum_pct"]])

assert "is_declining_label" not in queue.columns
assert queue["reason_code"].nunique() == 1
assert queue["action_label"].nunique() == 1
assert len(top20) == 20
assert OUTPUT_PATH.exists()
print("All checks passed.")

,baseline_rank,impressions_feature_30d,momentum_pct
16,17,2214.0,-93.403948


All checks passed.


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled: markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime > Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/`: then submit your repo URL on the card. Done.